# 02 — Functions, Classes, and Python Core

    - Design clear functions and small classes
- Avoid Python state and scope traps
- Stream large AI datasets lazily

    **Estimated study/practice time:** 3 hours  
    **Prerequisites:** Notebook 01

    **Priority legend:** 🔴 **Must Know** · 🟠 **Important** · 🟢 **Good to Know**

    ## Table of contents

    1. [Functions](#functions)
2. [Scope and first-class functions](#scope-and-first-class-functions)
3. [Classes and composition](#classes-and-composition)
4. [Typing and errors](#typing-and-errors)
5. [Generators](#generators)
6. [Practice](#practice)

    > Active-practice rule: run examples, predict output before revealing it, complete every `TODO`, then compare with the solution below it.


## Functions

🔴 **Must Know**

**Concept.** Functions package reusable behavior; parameters belong to the definition and arguments to the call.  
**Why it matters.** AI pipelines are clearer when validation, preprocessing, retrieval, and scoring are small testable functions.  
**Syntax.** `def name(required, default=..., *args, **kwargs): return value`.  
**How Python executes it.** Calling creates a local frame; defaults were already evaluated when `def` executed.  
**Complexity.** A function call is O(1) overhead plus the body’s work.  
**Common interview mistake.** Forgetting that no explicit `return` means `None`.  
**Interview question.** Parameters versus arguments? Names in a definition versus values supplied at a call.  
**Exercise/solution.** Complete *keyword configuration* and *independent defaults* below.


In [ ]:
def score(text: str, weight: float = 1.0, *features: float, **metadata: str) -> float:
    # Return a transparent toy score.
    print(metadata)
    return len(text) * weight + sum(features)

print(score("rag", 0.5, 1.0, source="docs"))


### 🔴 Mutable default argument trap

`items=[]` is created once, so calls share state. Use `None`, then allocate inside. **Interview question:** When are default expressions evaluated? Once, at function definition time.


In [ ]:
def bad_add(item, items=[]):
    items.append(item)
    return items

def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

print(bad_add("a"), bad_add("b"))
assert add_item("a") == ["a"] and add_item("b") == ["b"]


## Scope and first-class functions

🟠 **Important** — **Concept:** LEGB lookup order is Local → Enclosing → Global → Built-in, and functions are first-class objects. **Why it matters:** callbacks, sorting keys, decorators, and ML pipeline steps pass behavior around. **Syntax:** `nonlocal name`, `global name`, `apply(fn, value)`, and short `lambda x: ...`. **How Python executes it:** name lookup follows LEGB; closures retain enclosing references. **Complexity:** passing a function is O(1); calling it adds its body’s cost. **Common mistake:** rebinding an enclosing name without `nonlocal`. **Interview question:** when is `def` clearer than lambda? When logic needs a meaningful name or multiple steps. **Exercise/solution:** see *typed normalization*.


In [ ]:
threshold = 0.8
def make_filter(minimum):
    calls = 0
    def keep(prediction):
        nonlocal calls
        calls += 1
        return prediction["score"] >= minimum
    return keep

def apply(fn, values):
    return [fn(value) for value in values]

keep = make_filter(threshold)
predictions = [{"label":"cat", "score":.9}, {"label":"dog", "score":.6}]
print(list(filter(keep, predictions)))
print(sorted(predictions, key=lambda item: item["score"], reverse=True))


## Classes and composition

🔴 **Must Know** — **Concept:** a class defines behavior; instances own state, while class variables are shared. **Why it matters:** model configuration and RAG services benefit from explicit interfaces and replaceable components. **Syntax:** `class`, `__init__`, `self`, methods, inheritance with `super()`, composition through constructor arguments, and `@dataclass`. **How Python executes it:** attribute lookup checks the instance and then its class hierarchy. **Complexity:** ordinary attribute lookup is average O(1); method bodies determine overall cost. **Common mistake:** placing a mutable list on the class when each instance needs its own list. **Interview question:** composition versus inheritance? Prefer composition for a *has-a* relationship. **Exercise/solution:** complete *dataclass validation*.


In [ ]:
from dataclasses import dataclass, field

class BaseModel:
    def __init__(self, name: str): self.name = name
    def predict(self, text: str) -> str: raise NotImplementedError

class LLMModel(BaseModel):
    def __init__(self, name: str, temperature: float = 0.0):
        super().__init__(name)
        self.temperature = temperature
    def predict(self, text: str) -> str: return f"{self.name}: {text}"

@dataclass
class Chunk:
    text: str
    page: int
    source: str
    tags: list[str] = field(default_factory=list)  # new list per instance

class RAGService:
    def __init__(self, retriever, llm):
        self.retriever, self.llm = retriever, llm
    def answer(self, question):
        context = self.retriever(question)
        return self.llm.predict(f"{context}: {question}")

service = RAGService(lambda q: "retrieved context", LLMModel("toy"))
print(service.answer("What is RAG?"), Chunk("hello", 1, "guide"))

class BadRegistry:
    models = []  # shared by every instance: usually a bug

first_bad = BadRegistry()
second_bad = BadRegistry()
first_bad.models.append("shared-model")
assert second_bad.models == ["shared-model"]

class ModelRegistry:
    def __init__(self):
        self.models = []  # independent per instance

first_good, second_good = ModelRegistry(), ModelRegistry()
first_good.models.append("private-model")
assert second_good.models == []


## Typing and errors

🟠 **Important** — **Concept:** type hints document contracts; exceptions represent invalid operations; context managers own setup/cleanup. **Why it matters:** AI payloads, configurations, and preprocessing failures need readable contracts and actionable errors. **Syntax:** `str | None` (modern) or `Optional[str]`, `raise ValueError(...)`, `try/except/finally`, and `with`. **How Python executes it:** hints are normally not runtime enforcement; exceptions unwind until handled; `with` always calls its exit protocol. **Complexity:** annotations add no meaningful runtime cost by themselves. **Common mistake:** catching `Exception` and hiding unrelated bugs. **Interview question:** why catch specific exceptions? To handle only failures you understand. **Exercise/solution:** complete *safe division*.


In [ ]:
from typing import Optional

def find_user(user_id: int) -> str | None:
    return {1: "Alice"}.get(user_id)

def find_user_legacy_spelling(user_id: int) -> Optional[str]:
    return {1: "Alice"}.get(user_id)

def validate_chunk_size(chunk_size: int) -> None:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

try:
    validate_chunk_size(0)
except ValueError as error:
    print("validation:", error)
finally:
    print("validation finished")

from contextlib import nullcontext
with nullcontext("inference resource") as resource:
    print(resource)


### Context managers and basic decorators

`with open("file.txt") as file:` closes the file even if reading raises. The same protocol supports database transactions and inference contexts. A decorator receives a function and returns a replacement; learn the basic idea, but do not prioritize advanced decorator internals.

```python
def log_call(function):
    def wrapper(*args, **kwargs):
        print(function.__name__)
        return function(*args, **kwargs)
    return wrapper

@log_call
def embed(text): ...
```

**Interview question:** Why is `with` safer than manually opening and closing? Cleanup is guaranteed through the context-manager protocol, including exceptional exits.

## Generators

🟠 **Important** — **Concept:** an iterable can produce an iterator; an iterator tracks state; `yield` creates a generator that pauses between values. **Why it matters:** large AI datasets should be batched without materializing every batch. **Syntax:** `yield items[i:i+size]`. **How Python executes it:** a call returns a suspended generator; each `next` resumes until the next yield. **Complexity:** batching is O(n) total time and O(batch size) working space. **Common mistake:** returning a complete list and losing laziness. **Interview question:** generator versus list? Lazy one-pass production versus eager materialization. **Exercise/solution:** complete *lazy batches*.


In [ ]:
def batches(items, batch_size):
    if batch_size <= 0: raise ValueError("batch_size must be positive")
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

batch_iterator = batches(list(range(7)), 3)
print(iter(batch_iterator) is batch_iterator, list(batch_iterator))


## Practice


### Exercise — 🟢 Easy: independent defaults

        **Goal:** Fix a tag collector so calls do not share a list.

        **Example**

        ```text
        Input: separate calls
Output: separate lists
        ```

        <details><summary>Hint 1</summary>Use None.</details>
        <details><summary>Hint 2</summary>Allocate inside.</details>


In [ ]:
def add_tag(tag, tags=[]):
    # TODO: fix signature and body
    tags.append(tag)
    return tags


#### Solution — reveal only after attempting


In [ ]:
def add_tag(tag, tags=None):
    if tags is None: tags = []
    tags.append(tag)
    return tags
assert add_tag('a') == ['a'] and add_tag('b') == ['b']


**Complexity**

```text
Time: O(1) amortized; Space: O(1) per add
```

**Common mistake:** Checking `if not tags` replaces an intentionally empty list.


### Exercise — 🟢 Easy: keyword configuration

        **Goal:** Build a dict from required model plus keyword options.

        **Example**

        ```text
        Input: model_config('x', temperature=0)
Output: dict
        ```

        <details><summary>Hint 1</summary>Use dict unpacking.</details>
        <details><summary>Hint 2</summary>Keyword arguments arrive as a dict.</details>


In [ ]:
def model_config(model, **options):
    # TODO: Write your solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def model_config(model, **options):
    return {'model': model, **options}
assert model_config('x', temperature=0) == {'model':'x','temperature':0}


**Complexity**

        ```text
        Time: O(k)
Space: O(k)
        ```

        **Common mistake:** Mutating options unexpectedly.


### Exercise — 🟢 Easy: typed normalization

        **Goal:** Return lowercase stripped strings with a clear type hint.

        **Example**

        ```text
        Input: [' A ']
Output: ['a']
        ```

        <details><summary>Hint 1</summary>Map strip and lower.</details>
        <details><summary>Hint 2</summary>Return a new list.</details>


In [ ]:
def normalize_texts(texts):
    # TODO: add hints and implementation
    pass


#### Solution — reveal only after attempting


In [ ]:
def normalize_texts(texts: list[str]) -> list[str]:
    return [text.strip().lower() for text in texts]
assert normalize_texts([' A ']) == ['a']


**Complexity**

```text
Time: O(total characters); Space: O(total characters)
```

**Common mistake:** Claiming O(n) without defining whether n is strings or characters.


### Exercise — 🟢 Easy: safe division

        **Goal:** Raise ValueError when denominator is zero.

        **Example**

        ```text
        Input: 6,2
Output: 3
        ```

        <details><summary>Hint 1</summary>Validate first.</details>
        <details><summary>Hint 2</summary>Raise rather than return an ambiguous sentinel.</details>


In [ ]:
def safe_divide(a, b):
    # TODO: Write your solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def safe_divide(a: float, b: float) -> float:
    if b == 0: raise ValueError('denominator must be nonzero')
    return a / b
assert safe_divide(6, 2) == 3


**Complexity**

```text
Time: O(1); Space: O(1)
```

**Common mistake:** Catching every exception and hiding programming errors.


### Exercise — 🟢 Easy: lazy batches

        **Goal:** Yield chunks and reject nonpositive size.

        **Example**

        ```text
        Input: [1,2,3],2
Output: [1,2], then [3]
        ```

        <details><summary>Hint 1</summary>Use yield.</details>
        <details><summary>Hint 2</summary>Step range by size.</details>


In [ ]:
def lazy_batches(items, size):
    # TODO: Write your solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def lazy_batches(items, size):
    if size <= 0: raise ValueError('size must be positive')
    for i in range(0, len(items), size): yield items[i:i+size]
assert list(lazy_batches([1,2,3],2)) == [[1,2],[3]]


**Complexity**

```text
Time: O(n); extra Space: O(size)
```

**Common mistake:** Returning a complete list defeats laziness.


### Exercise — 🟢 Easy: dataclass validation

        **Goal:** Create an immutable ModelConfig with name and temperature.

        **Example**

        ```text
        Input: ModelConfig('demo')
Output: a result satisfying the stated goal
        ```

        <details><summary>Hint 1</summary>Use @dataclass(frozen=True).</details>
        <details><summary>Hint 2</summary>Provide a default.</details>


In [ ]:
# TODO: define ModelConfig


#### Solution — reveal only after attempting


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class ModelConfig:
    name: str
    temperature: float = 0.0
config = ModelConfig('demo')
assert config.temperature == 0.0


**Complexity**

```text
Time: O(1); Space: O(1)
```

**Common mistake:** Using a mutable shared class attribute.


## Key takeaways

    - Defaults are evaluated once.
- Composition keeps AI components replaceable.
- Generators stream work lazily.

    ## Mistakes to avoid

    - Mutable defaults/class variables.
- Broad exception swallowing.
- Overusing inheritance or lambdas.

    ## 10 rapid-fire interview questions

    1. **What does `b = a` do for a list?** <details><summary>Answer</summary>Aliases the same object.</details>
2. **What does a shallow copy share?** <details><summary>Answer</summary>Nested referenced objects.</details>
3. **What does deepcopy do?** <details><summary>Answer</summary>Recursively copies the reachable object graph.</details>
4. **When are default arguments evaluated?** <details><summary>Answer</summary>Once, when `def` executes.</details>
5. **Safe mutable-default pattern?** <details><summary>Answer</summary>Default to `None`, allocate inside.</details>
6. **What is LEGB?** <details><summary>Answer</summary>Local, Enclosing, Global, Built-in scope lookup.</details>
7. **What does `nonlocal` rebind?** <details><summary>Answer</summary>A name in an enclosing function scope.</details>
8. **What does `global` rebind?** <details><summary>Answer</summary>A module-level name.</details>
9. **Are functions first-class in Python?** <details><summary>Answer</summary>Yes: assign, pass, and return them.</details>
10. **When prefer `def` over lambda?** <details><summary>Answer</summary>When logic needs a name, statements, or explanation.</details>

    ## Readiness checklist

    - [ ] I can explain LEGB.
- [ ] I can write a generator.
- [ ] I can model a service with composition.
